# EXPERIMENT: EXP-RESNET50-SCRATCH-5SEEDS-004

## Experiment information
| Field | Value |
|---|---|
| Model | ResNet50 |
| Training mode | Scratch; weights=None; all backbone layers trainable |
| Task | 5-class cashew leaf classification |
| Dataset | Cashew_dataV04; existing CashewData_Split folders |
| Seeds | 42, 123, 2026, 3407, 7777 |
| Framework / GPU / date | Recorded at execution in environment.json |
| Standard | STANDARD_TRAINING_NOTEBOOK_FORMAT (1).md |

## Research objective
Measure ResNet50 scratch-training performance and variation across five fixed seeds.
The refactor preserves the architecture, training hyperparameters, class order and dataset splits.
No performance result is claimed before execution. EXP-004 uses a new output folder to preserve EXP-003 results.

## Locked dataset
| Class | Train | Validation | Test | Total |
|---|---:|---:|---:|---:|
| anthracnose | 945 | 294 | 122 | 1361 |
| healthy | 806 | 225 | 118 | 1149 |
| leaf_miner | 893 | 249 | 132 | 1274 |
| not_cashew_leaf | 1101 | 314 | 157 | 1572 |
| red_rust | 1077 | 320 | 158 | 1555 |
| **Total** | **4822** | **1402** | **687** | **6911** |

All seeds use the same existing split. This notebook never repartitions, deletes or relabels dataset files.
The audit reads files only to check integrity and counts; test predictions and evaluation require
`RUN_FINAL_TEST=True`, which is **False by default**. Select all settings using validation only.
Report all seeds as mean ± sample standard deviation (ddof=1); do not select a seed on test performance.

## Execution and outputs
Restart the kernel and Run All to train and validate. Results are isolated under seed folders with
checkpoints, logs, metrics, figures, predictions, model summaries and per-seed configuration.
Best checkpoints remain weights-only; rebuild with build_resnet50_model(seed) before loading them.
Both a full result archive (including checkpoints) and a lightweight analysis archive are produced.

After freezing the configuration, set RUN_FINAL_TEST=True and rerun the final-test section onward
in the same kernel. For a fresh kernel, also set RUN_TRAINING=False and initialize all earlier cells
against the same experiment output directory. Existing training outputs cannot be overwritten.
Manual inference is optional and uses predefined seed 42. A locked-test run is INCOMPLETE under
the standard's final-experiment definition, even when training and validation are complete.


In [ ]:
# ============================================================
# 01. IMPORT LIBRARIES
# ============================================================

import os
import sys
import io
import gc
import json
import time
import random
import shutil
import platform
import zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    ConfusionMatrixDisplay,
)

from IPython.display import display
import hashlib
import copy

print("Imports completed.")


In [ ]:
# ============================================================
# 02. ENVIRONMENT INFORMATION
# ============================================================

print(f"Date              : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python            : {sys.version.split()[0]}")
print(f"TensorFlow        : {tf.__version__}")
print(f"Keras             : {keras.__version__ if hasattr(keras, '__version__') else 'tf.keras'}")
print(f"Platform          : Kaggle")
print(f"OS                : {platform.platform()}")

gpus = tf.config.list_physical_devices("GPU")
print(f"GPU Count         : {len(gpus)}")

for i, gpu in enumerate(gpus):
    print(f"GPU {i}             : {gpu}")

if len(gpus) >= 2:
    strategy = tf.distribute.MirroredStrategy()
elif len(gpus) == 1:
    strategy = tf.distribute.OneDeviceStrategy(device="/GPU:0")
else:
    strategy = tf.distribute.OneDeviceStrategy(device="/CPU:0")

print(f"Strategy          : {strategy.__class__.__name__}")
print(f"Replicas          : {strategy.num_replicas_in_sync}")

build_info = tf.sysconfig.get_build_info()
environment = {
    "date": datetime.now().isoformat(), "python": sys.version,
    "tensorflow": tf.__version__, "keras": getattr(keras, "__version__", "tf.keras"),
    "platform": platform.platform(), "runtime": "Kaggle",
    "cuda": build_info.get("cuda_version"), "cudnn": build_info.get("cudnn_version"),
    "gpu_devices": [tf.config.experimental.get_device_details(g).get("device_name", str(g)) for g in gpus],
    "gpu_count": len(gpus), "strategy": strategy.__class__.__name__,
    "replicas": strategy.num_replicas_in_sync,
}
print(json.dumps(environment, indent=2))


In [ ]:
# ============================================================
# 03. CONFIGURATION
# ============================================================

EXPERIMENT_ID = "EXP-RESNET50-SCRATCH-5SEEDS-004"
DATASET_VERSION = "Cashew_dataV04"

EXPECTED_CLASSES = [
    "anthracnose",
    "healthy",
    "leaf_miner",
    "not_cashew_leaf",
    "red_rust",
]

EXPECTED_COUNTS = {
    "anthracnose": {
        "train": 945,
        "val": 294,
        "test": 122,
        "total": 1361,
    },

    "healthy": {
        "train": 806,
        "val": 225,
        "test": 118,
        "total": 1149,
    },

    "leaf_miner": {
        "train": 893,
        "val": 249,
        "test": 132,
        "total": 1274,
    },

    "not_cashew_leaf": {
        "train": 1101,
        "val": 314,
        "test": 157,
        "total": 1572,
    },

    "red_rust": {
        "train": 1077,
        "val": 320,
        "test": 158,
        "total": 1555,
    },
}

EXPECTED_TOTALS = {
    "train": 4822,
    "val": 1402,
    "test": 687,
    "total": 6911,
}
STRICT_DATASET_CHECK = True

# ------------------------------------------------------------
# 5-SEED PROTOCOL
# ------------------------------------------------------------

SEEDS = [42, 123, 2026, 3407, 7777]
DEMO_SEED = 42

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

IMG_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 50
INITIAL_LR = 1e-3

MODEL_NAME = "ResNet50"
TRAINING_MODE = "Scratch"
PRETRAINED_WEIGHTS = None

HEAD_UNITS = 512
HEAD_DROPOUT = 0.40

EARLY_STOPPING_PATIENCE = 8
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.3
MIN_LR = 1e-6

# ------------------------------------------------------------
# LOCKED TEST
# ------------------------------------------------------------

RUN_FINAL_TEST = False

RUN_TRAINING = True
RUN_MANUAL_DEMO = False
MEMBER = ""  # Fill in the experiment author's name before training.
OPTIMIZER = "Adam"
LOSS = "sparse_categorical_crossentropy"
TRAINING_METRICS = ["accuracy"]
MONITOR = "val_loss"
MONITOR_MODE = "min"
RESCALE_FACTOR = 1.0 / 255.0
FINE_TUNE = False  # Scratch training, not fine-tuning pretrained weights.
BACKBONE_TRAINABLE = True
AUGMENTATION_CONFIG = {
    "horizontal_flip": True, "vertical_flip": False,
    "rotation_factor": 0.04, "zoom_factor": 0.05,
    "translation_factor": 0.03, "contrast_factor": 0.08,
    "brightness": False, "hue_shift": False,
}
DATASET_OVERRIDE = None  # Explicit path required if multiple candidates exist.
ERROR_SAMPLE_COUNT = 20
SAMPLE_COUNT_PER_OUTCOME = 5
REGISTRY_PATH = Path("/kaggle/working/EXPERIMENT_REGISTRY.csv")

# ------------------------------------------------------------
# STORAGE
# ------------------------------------------------------------

# Chỉ lưu best weights để giảm mạnh dung lượng.
# Không lưu last.keras.
SAVE_BEST_WEIGHTS_ONLY = True

OUT_DIR = Path("/kaggle/working") / EXPERIMENT_ID
AGGREGATE_DIR = OUT_DIR / "aggregate"

if RUN_TRAINING and OUT_DIR.exists() and any(OUT_DIR.iterdir()):
    raise FileExistsError(f"Results already exist: {OUT_DIR}. Use a new experiment ID or RUN_TRAINING=False.")
if not RUN_TRAINING and not (OUT_DIR / "experiment_config.json").exists():
    raise FileNotFoundError("Evaluation-only mode requires a completed training run and its saved config.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATE_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment ID    :", EXPERIMENT_ID)
print("Dataset version  :", DATASET_VERSION)
print("Seeds            :", SEEDS)
print("Image size       :", IMG_SIZE)
print("Batch size       :", BATCH_SIZE)
print("Max epochs       :", MAX_EPOCHS)
print("Initial LR       :", INITIAL_LR)
print("RUN_FINAL_TEST   :", RUN_FINAL_TEST)


In [ ]:
# ============================================================
# 03a. REPRODUCIBILITY
# ============================================================

def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Đồng bộ Python / NumPy / TensorFlow theo Keras.
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass

    print(f"Global seed set to: {seed}")


try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic operations: ENABLED")
except Exception as e:
    print("Deterministic operations: unavailable")
    print("Reason:", e)


In [ ]:
# ============================================================
# 04. LOAD DATASET
# ============================================================

KAGGLE_INPUT = Path("/kaggle/input")
TARGET_FOLDER_NAME = "CashewData_Split"

candidates = [
    p for p in KAGGLE_INPUT.rglob(TARGET_FOLDER_NAME)
    if p.is_dir()
    and (p / "train").exists()
    and (p / "val").exists()
    and (p / "test").exists()
]

if not candidates:
    raise FileNotFoundError(
        f"Không tìm thấy folder '{TARGET_FOLDER_NAME}' có đủ train/val/test trong /kaggle/input."
    )

print("Dataset candidates:")
for i, p in enumerate(candidates):
    print(f"[{i}] {p}")

# Nếu Kaggle chỉ attach một dataset phù hợp, candidate 0 là đúng.
if DATASET_OVERRIDE is not None:
    DATASET_PATH = Path(DATASET_OVERRIDE)
    if DATASET_PATH not in candidates:
        raise ValueError("DATASET_OVERRIDE must be one of the detected split roots.")
elif len(candidates) == 1:
    DATASET_PATH = candidates[0]
else:
    raise ValueError("Multiple datasets found. Set DATASET_OVERRIDE explicitly; do not silently switch test sets.")

TRAIN_DIR = DATASET_PATH / "train"
VAL_DIR = DATASET_PATH / "val"
TEST_DIR = DATASET_PATH / "test"

print("\nSelected dataset:")
print("DATASET_PATH :", DATASET_PATH)
print("TRAIN_DIR    :", TRAIN_DIR)
print("VAL_DIR      :", VAL_DIR)
print("TEST_DIR     :", TEST_DIR)


## 05. Dataset statistics and read-only integrity audit
Verify the existing V04 class counts and image integrity. The audit does not repair or modify any source image.


In [ ]:
# ============================================================
# 05. DATASET STATISTICS
# ============================================================

VALID_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"
}


def audit_split(split_name, split_dir):
    valid_rows = []
    bad_rows = []
    unknown_rows = []

    found_classes = sorted([
        p.name for p in split_dir.iterdir()
        if p.is_dir()
    ])

    if found_classes != sorted(EXPECTED_CLASSES):
        raise ValueError(
            f"Class folders của {split_name} không đúng.\n"
            f"Expected: {sorted(EXPECTED_CLASSES)}\n"
            f"Found   : {found_classes}"
        )

    for label, class_name in enumerate(EXPECTED_CLASSES):
        class_dir = split_dir / class_name

        for path in sorted(class_dir.rglob("*")):
            if not path.is_file():
                continue

            ext = path.suffix.lower()

            if ext not in VALID_EXTENSIONS:
                unknown_rows.append({
                    "split": split_name,
                    "class": class_name,
                    "path": str(path),
                    "extension": ext,
                })
                continue

            try:
                with Image.open(path) as img:
                    img.verify()

                with Image.open(path) as img:
                    img.convert("RGB").load()

                valid_rows.append({
                    "split": split_name,
                    "class": class_name,
                    "label": label,
                    "path": str(path),
                })

            except Exception as e:
                bad_rows.append({
                    "split": split_name,
                    "class": class_name,
                    "path": str(path),
                    "error": str(e),
                })

    return valid_rows, bad_rows, unknown_rows


all_valid = []
all_bad = []
all_unknown = []

for split_name, split_dir in {
    "train": TRAIN_DIR,
    "val": VAL_DIR,
    "test": TEST_DIR,
}.items():

    print(f"Checking {split_name}...")

    valid_rows, bad_rows, unknown_rows = audit_split(
        split_name,
        split_dir
    )

    all_valid.extend(valid_rows)
    all_bad.extend(bad_rows)
    all_unknown.extend(unknown_rows)


valid_df = pd.DataFrame(all_valid)
bad_df = pd.DataFrame(all_bad)
unknown_df = pd.DataFrame(all_unknown)

print("\n" + "=" * 70)
print("VALID IMAGES   :", len(valid_df))
print("INVALID IMAGES :", len(bad_df))
print("UNKNOWN FILES  :", len(unknown_df))
print("=" * 70)

if len(bad_df):
    display(bad_df)

if len(unknown_df):
    display(unknown_df)

bad_df.to_csv(OUT_DIR / "invalid_images.csv", index=False)
unknown_df.to_csv(OUT_DIR / "unknown_files.csv", index=False)

if len(bad_df):
    raise ValueError("Dataset còn ảnh lỗi. Hãy clean dataset trước khi train.")

if len(unknown_df):
    raise ValueError("Dataset có file extension lạ. Hãy kiểm tra trước khi train.")


stats_df = (
    valid_df.groupby(["class", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(EXPECTED_CLASSES)
)

for col in ["train", "val", "test"]:
    if col not in stats_df.columns:
        stats_df[col] = 0

stats_df = stats_df[["train", "val", "test"]]
stats_df["total"] = stats_df.sum(axis=1)

display(stats_df)

actual_totals = {
    "train": int(stats_df["train"].sum()),
    "val": int(stats_df["val"].sum()),
    "test": int(stats_df["test"].sum()),
    "total": int(stats_df["total"].sum()),
}

mismatches = []

for class_name in EXPECTED_CLASSES:
    for split_name in ["train", "val", "test", "total"]:

        actual = int(stats_df.loc[class_name, split_name])
        expected = int(EXPECTED_COUNTS[class_name][split_name])

        if actual != expected:
            mismatches.append({
                "class": class_name,
                "split": split_name,
                "actual": actual,
                "expected": expected,
            })

for split_name in ["train", "val", "test", "total"]:
    if actual_totals[split_name] != EXPECTED_TOTALS[split_name]:
        mismatches.append({
            "class": "TOTAL",
            "split": split_name,
            "actual": actual_totals[split_name],
            "expected": EXPECTED_TOTALS[split_name],
        })

if mismatches:
    mismatch_df = pd.DataFrame(mismatches)
    display(mismatch_df)

    if STRICT_DATASET_CHECK:
        raise ValueError(
            "Dataset count không khớp Cashew_dataV04 đã chốt."
        )
else:
    print("Dataset counts match Cashew_dataV04 exactly.")
    print(actual_totals)


# Save manifest để biết chính xác file nào thuộc split nào.
valid_df.to_csv(
    OUT_DIR / "dataset_manifest.csv",
    index=False
)

stats_df.to_csv(OUT_DIR / "dataset_statistics.csv")

ax = stats_df[["train", "val", "test"]].plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Cashew_dataV04 - Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of Images")

plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plt.savefig(
    AGGREGATE_DIR / "class_distribution.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


## 06. Preprocessing
ResNet50 is trained from scratch (weights=None). Decode RGB, resize to 224×224 with TensorFlow bilinear interpolation, then apply Rescaling(1/255) inside the model. Do not apply ImageNet preprocess_input or normalize a second time. Validation and test preserve file order and use inference mode without augmentation.


In [ ]:
# ============================================================
# 06. PREPROCESSING
# ============================================================

CLASS_NAMES = EXPECTED_CLASSES
NUM_CLASSES = len(CLASS_NAMES)
AUTOTUNE = tf.data.AUTOTUNE


def split_arrays(split_name):
    df = (
        valid_df[valid_df["split"] == split_name]
        .copy()
        .reset_index(drop=True)
    )

    return (
        df["path"].to_numpy(),
        df["label"].to_numpy(dtype=np.int32),
        df
    )


train_paths, train_labels, train_meta = split_arrays("train")
val_paths, val_labels, val_meta = split_arrays("val")
test_paths, test_labels, test_meta = split_arrays("test")


def decode_image(path, label):
    image_bytes = tf.io.read_file(path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        method="bilinear"
    )

    return tf.cast(image, tf.float32), label


## 07. Data augmentation
Training only: horizontal flip, rotation 0.04, zoom ±0.05, translation 0.03, contrast 0.08. No vertical flip, brightness or hue augmentation. Values are declared in Configuration and saved in experiment_config.json.


In [ ]:
# ============================================================
# 07. DATA AUGMENTATION
# ============================================================

def build_data_augmentation(seed):
    return keras.Sequential(
        ([layers.RandomFlip(
                ("horizontal_and_vertical" if AUGMENTATION_CONFIG["vertical_flip"] else "horizontal")
                if AUGMENTATION_CONFIG["horizontal_flip"] else "vertical",
                seed=seed + 1
            )] if AUGMENTATION_CONFIG["horizontal_flip"] or AUGMENTATION_CONFIG["vertical_flip"] else []) + [
            layers.RandomRotation(
                factor=AUGMENTATION_CONFIG["rotation_factor"],
                seed=seed + 2
            ),

            layers.RandomZoom(
                height_factor=(-AUGMENTATION_CONFIG["zoom_factor"], AUGMENTATION_CONFIG["zoom_factor"]),
                width_factor=(-AUGMENTATION_CONFIG["zoom_factor"], AUGMENTATION_CONFIG["zoom_factor"]),
                seed=seed + 3
            ),

            layers.RandomTranslation(
                height_factor=AUGMENTATION_CONFIG["translation_factor"],
                width_factor=AUGMENTATION_CONFIG["translation_factor"],
                seed=seed + 4
            ),

            layers.RandomContrast(
                factor=AUGMENTATION_CONFIG["contrast_factor"],
                seed=seed + 5
            ),
        ],
        name=f"data_augmentation_seed_{seed}"
    )


In [ ]:
# ============================================================
# 08. DATA PIPELINE
# ============================================================

def make_dataset(paths, labels, seed, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(paths),
            seed=seed,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        decode_image,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(
        BATCH_SIZE,
        drop_remainder=False
    )

    ds = ds.prefetch(AUTOTUNE)

    return ds


print("Fixed split:")
print("Train :", len(train_paths))
print("Val   :", len(val_paths))
print("Test  :", len(test_paths))

print("\nClass mapping:")
for idx, class_name in enumerate(CLASS_NAMES):
    print(f"{idx}: {class_name}")
print("Number of classes:", NUM_CLASSES)


In [ ]:
# ============================================================
# 08a. TRAIN-ONLY AUGMENTATION PREVIEW
# ============================================================

preview_seed = SEEDS[0]

preview_ds = make_dataset(
    train_paths,
    train_labels,
    seed=preview_seed,
    training=True
)

preview_aug = build_data_augmentation(preview_seed)

for images, labels in preview_ds.take(1):

    n = min(6, int(images.shape[0]))

    plt.figure(figsize=(12, 6))

    for i in range(n):

        plt.subplot(2, 6, i + 1)
        plt.imshow(
            tf.cast(
                tf.clip_by_value(images[i], 0, 255),
                tf.uint8
            )
        )
        plt.title(CLASS_NAMES[int(labels[i])])
        plt.axis("off")

        aug_img = preview_aug(
            tf.expand_dims(images[i], 0),
            training=True
        )[0]

        plt.subplot(2, 6, i + 7)
        plt.imshow(
            tf.cast(
                tf.clip_by_value(aug_img, 0, 255),
                tf.uint8
            )
        )
        plt.title("Augmented")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

del preview_ds, preview_aug
gc.collect()


In [ ]:
# ============================================================
# 09. BUILD MODEL
# ============================================================

def build_resnet50_model(seed):

    set_global_seed(seed)

    augmentation = build_data_augmentation(seed)

    inputs = keras.Input(
        shape=(IMG_SIZE, IMG_SIZE, 3),
        name="image"
    )

    # Augmentation chỉ hoạt động khi model ở training mode.
    x = augmentation(inputs)

    # Scratch training: normalize ảnh về [0,1].
    x = layers.Rescaling(
        RESCALE_FACTOR,
        name="rescale_0_1"
    )(x)

    backbone = tf.keras.applications.ResNet50(
        include_top=False,
        weights=PRETRAINED_WEIGHTS,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    backbone.trainable = BACKBONE_TRAINABLE

    # ========================================================
    # QUAN TRỌNG:
    # Không ép training=True.
    # ========================================================
    x = backbone(x)

    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        HEAD_UNITS,
        activation="relu",
        name="classifier_dense"
    )(x)

    x = layers.BatchNormalization(
        name="classifier_bn"
    )(x)

    x = layers.Dropout(
        HEAD_DROPOUT,
        seed=seed + 6,
        name="classifier_dropout"
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        name="predictions"
    )(x)

    return keras.Model(
        inputs=inputs,
        outputs=outputs,
        name=f"ResNet50_Scratch_seed_{seed}"
    )


print("Defined:")
print("- build_data_augmentation(seed)")
print("- build_resnet50_model(seed)")


In [ ]:
# ============================================================
# 10. MODEL SUMMARY AND SANITY CHECK
# ============================================================

# Cell này giúp phát hiện lỗi function trước khi mất hàng giờ train.

tf.keras.backend.clear_session()
gc.collect()

sanity_seed = SEEDS[0]

with strategy.scope():
    sanity_model = build_resnet50_model(sanity_seed)

expected_output_shape = (None, NUM_CLASSES)

print("Model name        :", sanity_model.name)
print("Output shape      :", sanity_model.output_shape)
print("Total parameters  :", sanity_model.count_params())

assert sanity_model.output_shape == expected_output_shape
assert any(layer.name == "resnet50" for layer in sanity_model.layers)

print("\nSANITY CHECK PASSED.")
print("Training cell sẽ gọi đúng build_resnet50_model(seed).")

sanity_model.summary()
summary_lines = []
sanity_model.summary(print_fn=lambda line, **kwargs: summary_lines.append(line))
(OUT_DIR / "model_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")
print("Trainable parameters:", sum(int(np.prod(v.shape)) for v in sanity_model.trainable_weights))
print("Non-trainable parameters:", sum(int(np.prod(v.shape)) for v in sanity_model.non_trainable_weights))
del sanity_model
tf.keras.backend.clear_session()
gc.collect()


## 11. Compile configuration
Adam, initial learning rate 1e-3; sparse categorical crossentropy; accuracy. Every seed gets a new optimizer. Compilation remains inside the distribution strategy scope.


In [ ]:
# ============================================================
# 11. COMPILE MODEL
# ============================================================

def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.get({"class_name": OPTIMIZER, "config": {"learning_rate": INITIAL_LR}}),
        loss=LOSS, metrics=TRAINING_METRICS,
    )


In [ ]:
# ============================================================
# 12. CALLBACKS
# ============================================================

class LearningRateRecorder(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))

    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            logs["learning_rate"] = self.epoch_lr


def build_callbacks(best_weights_path, log_dir):
    return [
        LearningRateRecorder(),
        keras.callbacks.ModelCheckpoint(str(best_weights_path), monitor=MONITOR,
            mode=MONITOR_MODE, save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor=MONITOR, mode=MONITOR_MODE,
            patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor=MONITOR, mode=MONITOR_MODE,
            factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        keras.callbacks.CSVLogger(str(log_dir / "training_log.csv")),
    ]

print("Checkpoint: minimum val_loss; best weights only")
print("EarlyStopping: minimum val_loss; patience", EARLY_STOPPING_PATIENCE, "; restore best weights")
print("ReduceLR: minimum val_loss; patience", REDUCE_LR_PATIENCE,
      "; factor", REDUCE_LR_FACTOR, "; minimum", MIN_LR)


In [ ]:
# ============================================================
# 13. SAVE EXPERIMENT CONFIG
# ============================================================

experiment_config = {
    "experiment_id": EXPERIMENT_ID,
    "created_at": datetime.now().isoformat(),

    "dataset": {
        "version": DATASET_VERSION,
        "path": str(DATASET_PATH),
        "classes": CLASS_NAMES,
        "train_images": len(train_paths),
        "val_images": len(val_paths),
        "test_images": len(test_paths),
        "total_images": len(valid_df),
        "test_set_locked": True,
    },

    "model": {
        "name": MODEL_NAME,
        "training_mode": TRAINING_MODE,
        "weights": PRETRAINED_WEIGHTS,
        "include_top": False,
        "input_size": [IMG_SIZE, IMG_SIZE, 3],
        "head_units": HEAD_UNITS,
        "head_dropout": HEAD_DROPOUT,
        "backbone_call": "x = backbone(x)",
        "forced_training_true": False,
    },

    "training": {
        "seeds": SEEDS,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "optimizer": "Adam",
        "initial_learning_rate": INITIAL_LR,
        "loss": "SparseCategoricalCrossentropy",
        "class_weight": None,
    },

    "augmentation": AUGMENTATION_CONFIG,

    "callbacks": {
        "checkpoint_monitor": "val_loss",
        "checkpoint_mode": "min",
        "checkpoint_type": "weights_only",
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "reduce_lr_patience": REDUCE_LR_PATIENCE,
        "reduce_lr_factor": REDUCE_LR_FACTOR,
        "min_lr": MIN_LR,
    },

    "test_protocol": {
        "run_final_test": RUN_FINAL_TEST,
        "same_locked_test_for_all_seeds": True,
        "do_not_select_best_seed_on_test": True,
        "summary": "mean ± sample standard deviation (ddof=1)",
    },

    "hardware": {
        "gpu_count": len(gpus),
        "strategy": strategy.__class__.__name__,
        "replicas": strategy.num_replicas_in_sync,
    },
}


experiment_config["member"] = MEMBER
experiment_config["environment"] = environment
experiment_config["model"].update({"fine_tune": FINE_TUNE, "backbone_trainable": BACKBONE_TRAINABLE,
    "preprocessing": {"resize": "bilinear", "rescale_factor": RESCALE_FACTOR}})
experiment_config["training"].update({"optimizer": OPTIMIZER, "loss": LOSS, "metrics": TRAINING_METRICS})
experiment_config["callbacks"].update({"checkpoint_monitor": MONITOR, "checkpoint_mode": MONITOR_MODE,
    "save_best_only": True, "save_weights_only": True, "restore_best_weights": True,
    "early_stopping_monitor": MONITOR, "reduce_lr_monitor": MONITOR})

# Fingerprint existing file identities AND bytes; never alter the split.
split_fingerprints = {}
for split in ("train", "val", "test"):
    digest = hashlib.sha256()
    for path in valid_df.loc[valid_df["split"] == split, "path"]:
        source_path = Path(path)
        digest.update(source_path.relative_to(DATASET_PATH).as_posix().encode("utf-8") + b"\0")
        with source_path.open("rb") as source_file:
            for block in iter(lambda: source_file.read(1024 * 1024), b""):
                digest.update(block)
    split_fingerprints[split] = digest.hexdigest()
experiment_config["dataset"]["sha256_by_split"] = split_fingerprints

config_path = OUT_DIR / "experiment_config.json"
if config_path.exists():
    saved_config = json.loads(config_path.read_text(encoding="utf-8"))
    for key in ("dataset", "model", "training", "augmentation", "callbacks"):
        if saved_config[key] != experiment_config[key]:
            raise ValueError(f"Saved {key} differs from this notebook. Restore the frozen configuration.")
    experiment_config = saved_config
else:
    config_path.write_text(json.dumps(experiment_config, indent=2, ensure_ascii=False), encoding="utf-8")
(OUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Saved/verified:", config_path)


## 14–16. Train, save history and learning curves (five seeds)
The three standard stages run together for each seed to avoid retaining five models in GPU memory. Checkpoints are selected by minimum validation loss. Test inference is excluded. History, configuration, model summaries and learning-curve diagnostics are saved per seed.


In [ ]:
# ============================================================
# 14–16. TRAIN / SAVE HISTORY / LEARNING CURVES
# ============================================================

if RUN_TRAINING:
    training_rows = []
    validation_rows = []
    
    for run_index, seed in enumerate(SEEDS, start=1):
    
        print("\n" + "=" * 80)
        print(f"RUN {run_index}/{len(SEEDS)} - SEED {seed}")
        print("=" * 80)
    
        tf.keras.backend.clear_session()
        gc.collect()
        set_global_seed(seed)
    
        # --------------------------------------------------------
        # Folders
        # --------------------------------------------------------
    
        seed_dir = OUT_DIR / f"seed_{seed}"
    
        checkpoint_dir = seed_dir / "checkpoints"
        log_dir = seed_dir / "logs"
        figure_dir = seed_dir / "figures"
        validation_dir = seed_dir / "validation"
    
        for folder in [
            checkpoint_dir,
            log_dir,
            figure_dir,
            validation_dir,
        ]:
            folder.mkdir(parents=True, exist_ok=True)
    
        best_weights_path = (
            checkpoint_dir /
            f"RESNET50_SCRATCH_seed_{seed}_best.weights.h5"
        )
    
        if best_weights_path.exists() or (log_dir / "history.csv").exists():
            raise FileExistsError(f"Refusing to overwrite training outputs for seed {seed}.")
        seed_config = copy.deepcopy(experiment_config)
        seed_config["training"]["seed"] = seed
        (seed_dir / "experiment_config.json").write_text(json.dumps(seed_config, indent=2), encoding="utf-8")
    
        # --------------------------------------------------------
        # Datasets
        # --------------------------------------------------------
    
        train_ds = make_dataset(
            train_paths,
            train_labels,
            seed=seed,
            training=True
        )
    
        val_ds = make_dataset(
            val_paths,
            val_labels,
            seed=seed,
            training=False
        )
    
        # --------------------------------------------------------
        # Build + compile
        # --------------------------------------------------------
    
        with strategy.scope():
    
            # CHỈ dùng function này.
            model = build_resnet50_model(seed)
    
            compile_model(model)
    
        # --------------------------------------------------------
        # Model summary
        # --------------------------------------------------------
    
        summary_lines = []
    
        model.summary(
            print_fn=lambda line, **kwargs: summary_lines.append(line)
        )
    
        (
            seed_dir /
            "model_summary.txt"
        ).write_text(
            "\n".join(summary_lines),
            encoding="utf-8"
        )
    
        total_params = int(model.count_params())
    
        trainable_params = int(
            np.sum([
                np.prod(v.shape)
                for v in model.trainable_weights
            ])
        )
    
        non_trainable_params = int(
            total_params - trainable_params
        )
    
        # --------------------------------------------------------
        # Callbacks
        # --------------------------------------------------------
    
        callbacks = build_callbacks(best_weights_path, log_dir)
    
        # --------------------------------------------------------
        # Train
        # --------------------------------------------------------
    
        start_time = time.time()
    
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=MAX_EPOCHS,
            callbacks=callbacks,
            verbose=1,
        )
    
        training_time_seconds = time.time() - start_time
    
        # --------------------------------------------------------
        # History
        # --------------------------------------------------------
    
        history_df = pd.DataFrame(history.history)
    
        history_df.insert(
            0,
            "epoch",
            np.arange(1, len(history_df) + 1)
        )
    
        history_df.to_csv(
            log_dir / "history.csv",
            index=False
        )
    
        actual_epochs = len(history_df)
    
        best_epoch = int(
            np.argmin(history.history["val_loss"])
        ) + 1
    
        best_val_loss_history = float(
            np.min(history.history["val_loss"])
        )
    
        best_val_accuracy_history = float(
            history.history["val_accuracy"][best_epoch - 1]
        )
    
        # --------------------------------------------------------
        # Curves
        # --------------------------------------------------------
    
        plt.figure(figsize=(8, 5))
    
        plt.plot(
            history_df["epoch"],
            history_df["accuracy"],
            label="Train"
        )
    
        plt.plot(
            history_df["epoch"],
            history_df["val_accuracy"],
            label="Validation"
        )
    
        plt.axvline(
            best_epoch,
            linestyle="--",
            label=f"Best epoch = {best_epoch}"
        )
    
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title(f"ResNet50 Scratch - Seed {seed} - Accuracy")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
    
        plt.savefig(
            figure_dir / "accuracy_curve.png",
            dpi=200,
            bbox_inches="tight"
        )
    
        plt.close()
    
        plt.figure(figsize=(8, 5))
    
        plt.plot(
            history_df["epoch"],
            history_df["loss"],
            label="Train"
        )
    
        plt.plot(
            history_df["epoch"],
            history_df["val_loss"],
            label="Validation"
        )
    
        plt.axvline(
            best_epoch,
            linestyle="--",
            label=f"Best epoch = {best_epoch}"
        )
    
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"ResNet50 Scratch - Seed {seed} - Loss")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
    
        plt.savefig(
            figure_dir / "loss_curve.png",
            dpi=200,
            bbox_inches="tight"
        )
    
        plt.close()
    
        # --------------------------------------------------------
        # IMPORTANT:
        # Reload exactly the checkpoint chosen by minimum val_loss.
        # --------------------------------------------------------
    
        model.load_weights(best_weights_path)
    
        # --------------------------------------------------------
        # Validation evaluation
        # --------------------------------------------------------
    
        val_loss, val_accuracy = model.evaluate(
            val_ds,
            verbose=0
        )
    
        y_val_true = []
        y_val_prob = []
    
        for images, labels in val_ds:
    
            probs = model.predict(
                images,
                verbose=0
            )
    
            y_val_prob.append(probs)
            y_val_true.append(labels.numpy())
    
        y_val_true = np.concatenate(y_val_true)
        y_val_prob = np.concatenate(y_val_prob)
        y_val_pred = np.argmax(y_val_prob, axis=1)
        y_val_conf = np.max(y_val_prob, axis=1)
    
        val_metrics = {
            "seed": seed,
            "val_loss": float(val_loss),
            "val_accuracy": float(
                accuracy_score(y_val_true, y_val_pred)
            ),
            "val_macro_precision": float(
                precision_score(
                    y_val_true,
                    y_val_pred,
                    average="macro",
                    zero_division=0
                )
            ),
            "val_macro_recall": float(
                recall_score(
                    y_val_true,
                    y_val_pred,
                    average="macro",
                    zero_division=0
                )
            ),
            "val_macro_f1": float(
                f1_score(
                    y_val_true,
                    y_val_pred,
                    average="macro",
                    zero_division=0
                )
            ),
            "val_balanced_accuracy": float(
                balanced_accuracy_score(
                    y_val_true,
                    y_val_pred
                )
            ),
        }
    
        validation_rows.append(val_metrics)
    
        # --------------------------------------------------------
        # Validation classification report
        # --------------------------------------------------------
    
        val_report_df = pd.DataFrame(
            classification_report(
                y_val_true,
                y_val_pred,
                target_names=CLASS_NAMES,
                output_dict=True,
                zero_division=0,
            )
        ).transpose()
    
        val_report_df.to_csv(
            validation_dir /
            "classification_report.csv"
        )
    
        # --------------------------------------------------------
        # Validation confusion matrix
        # --------------------------------------------------------
    
        val_cm = confusion_matrix(
            y_val_true,
            y_val_pred
        )
    
        val_cm_norm = confusion_matrix(
            y_val_true,
            y_val_pred,
            normalize="true"
        )
    
        pd.DataFrame(
            val_cm,
            index=CLASS_NAMES,
            columns=CLASS_NAMES
        ).to_csv(
            validation_dir /
            "confusion_matrix.csv"
        )
    
        pd.DataFrame(
            val_cm_norm,
            index=CLASS_NAMES,
            columns=CLASS_NAMES
        ).to_csv(
            validation_dir /
            "confusion_matrix_normalized.csv"
        )
    
        fig, ax = plt.subplots(figsize=(8, 7))
    
        ConfusionMatrixDisplay(
            confusion_matrix=val_cm,
            display_labels=CLASS_NAMES
        ).plot(
            ax=ax,
            cmap="Blues",
            values_format="d",
            colorbar=False
        )
    
        plt.title(
            f"Validation Confusion Matrix - Seed {seed}"
        )
    
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
    
        plt.savefig(
            validation_dir /
            "confusion_matrix.png",
            dpi=200,
            bbox_inches="tight"
        )
    
        plt.close()
    
        # --------------------------------------------------------
        # Validation predictions
        # --------------------------------------------------------
    
        val_predictions_df = pd.DataFrame({
            "filename": val_meta["path"],
            "true_label": [
                CLASS_NAMES[int(i)]
                for i in y_val_true
            ],
            "predicted_label": [
                CLASS_NAMES[int(i)]
                for i in y_val_pred
            ],
            "confidence": y_val_conf,
            "correct": y_val_true == y_val_pred,
        })
    
        val_predictions_df.to_csv(
            validation_dir /
            "predictions.csv",
            index=False
        )
    
        # --------------------------------------------------------
        # Training summary
        # --------------------------------------------------------
    
        row = {
            "seed": seed,
            "actual_epochs": actual_epochs,
            "best_epoch": best_epoch,
            "best_val_loss_history": best_val_loss_history,
            "best_val_accuracy_history": best_val_accuracy_history,
            "checkpoint_val_loss": float(val_loss),
            "checkpoint_val_accuracy": float(val_metrics["val_accuracy"]),
            "checkpoint_val_macro_f1": float(val_metrics["val_macro_f1"]),
            "checkpoint_val_balanced_accuracy": float(
                val_metrics["val_balanced_accuracy"]
            ),
            "training_time_seconds": float(training_time_seconds),
            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "non_trainable_parameters": non_trainable_params,
            "best_weights_path": str(best_weights_path),
            "best_weights_size_mb": float(
                best_weights_path.stat().st_size /
                (1024 ** 2)
            ),
        }
    
        row["max_epochs"] = MAX_EPOCHS
        row["maximum_val_accuracy_history"] = float(history_df["val_accuracy"].max())
        lr_changes = history_df.loc[history_df["learning_rate"].diff() < 0, "epoch"].astype(int).tolist()
        last = history_df.iloc[-1]
        curve_notes = {
            "best_validation_epoch": best_epoch,
            "final_train_val_accuracy_gap": float(last["accuracy"] - last["val_accuracy"]),
            "final_val_loss_above_minimum": float(last["val_loss"] - history_df["val_loss"].min()),
            "epochs_using_reduced_lr": lr_changes,
            "early_stopping_epoch": actual_epochs if callbacks[2].stopped_epoch > 0 else None,
            "interpretation": "Inspect gap and post-minimum validation loss for overfitting; these diagnostics alone do not establish a cause.",
        }
        (log_dir / "learning_curve_analysis.json").write_text(json.dumps(curve_notes, indent=2), encoding="utf-8")
        print("Learning-curve diagnostics:", curve_notes)
        training_rows.append(row)
    
        (
            seed_dir /
            "training_summary.json"
        ).write_text(
            json.dumps(
                row,
                indent=2
            ),
            encoding="utf-8"
        )
    
        print("\nSeed summary:")
        print(json.dumps(row, indent=2))
    
        # --------------------------------------------------------
        # Cleanup
        # --------------------------------------------------------
    
        del model
        del train_ds
        del val_ds
        del history
    
        tf.keras.backend.clear_session()
        gc.collect()
    
    
    training_summary_df = pd.DataFrame(training_rows)
    validation_results_df = pd.DataFrame(validation_rows)
    
    training_summary_df.to_csv(
        AGGREGATE_DIR /
        "training_summary_5seeds.csv",
        index=False
    )
    
    validation_results_df.to_csv(
        AGGREGATE_DIR /
        "validation_results_5seeds.csv",
        index=False
    )
    
    print("\nTraining summary:")
    display(training_summary_df)
    
    print("\nValidation metrics:")
    display(validation_results_df)
else:
    training_summary_df = pd.read_csv(AGGREGATE_DIR / "training_summary_5seeds.csv")
    validation_results_df = pd.read_csv(AGGREGATE_DIR / "validation_results_5seeds.csv")
    if training_summary_df["seed"].tolist() != SEEDS or validation_results_df["seed"].tolist() != SEEDS:
        raise ValueError("Saved training/validation results must contain exactly the configured seeds.")
    print("Loaded existing training and validation results; training skipped.")


In [ ]:
# ============================================================
# 16a. VALIDATION MEAN ± STD
# ============================================================

validation_metrics_to_summarize = [
    "val_accuracy",
    "val_macro_precision",
    "val_macro_recall",
    "val_macro_f1",
    "val_balanced_accuracy",
]

validation_summary_rows = []

for metric in validation_metrics_to_summarize:

    values = validation_results_df[metric]

    validation_summary_rows.append({
        "metric": metric,
        "mean": float(values.mean()),
        "std": float(values.std(ddof=1)),
        "report": (
            f"{values.mean()*100:.2f} ± "
            f"{values.std(ddof=1)*100:.2f}%"
        ),
    })

validation_mean_std_df = pd.DataFrame(
    validation_summary_rows
)

display(validation_mean_std_df)

validation_mean_std_df.to_csv(
    AGGREGATE_DIR /
    "validation_mean_std_summary.csv",
    index=False
)

print("\nBest epoch across seeds:")
print(
    f"{training_summary_df['best_epoch'].mean():.2f} ± "
    f"{training_summary_df['best_epoch'].std(ddof=1):.2f}"
)


## 17–26. Locked final evaluation and analysis
Run only after all five seeds and the configuration are frozen. Reload the best validation checkpoint for each seed. The numbered helpers below implement the standard reporting stages; a single guarded loop runs them per seed to bound memory use. Timing measures full-pipeline batched prediction after warm-up, including decoding and host overhead; it is not single-image latency.


In [ ]:
# ============================================================
# 17. LOAD BEST MODEL BEFORE TEST
# ============================================================

def load_best_model(seed):
    checkpoint = OUT_DIR / f"seed_{seed}" / "checkpoints" / f"RESNET50_SCRATCH_seed_{seed}_best.weights.h5"
    if not checkpoint.exists():
        raise FileNotFoundError(checkpoint)
    tf.keras.backend.clear_session()
    gc.collect()
    with strategy.scope():
        model = build_resnet50_model(seed)
        model.load_weights(checkpoint)
        compile_model(model)
    return model, checkpoint


In [ ]:
# ============================================================
# 18. TEST EVALUATION
# ============================================================

def evaluate_test(model, dataset):
    loss, accuracy = model.evaluate(dataset, verbose=0)
    return float(loss), float(accuracy)


In [ ]:
# ============================================================
# 19. TEST PREDICTION
# ============================================================

def predict_test(model, dataset):
    _ = model.predict(dataset.take(1), verbose=0)  # Full batch for multi-GPU warm-up.
    start = time.perf_counter()
    probabilities = model.predict(dataset, verbose=0)
    elapsed = time.perf_counter() - start
    if probabilities.shape != (len(test_labels), NUM_CLASSES):
        raise ValueError("Prediction shape does not match the fixed test file list.")
    return probabilities, elapsed


In [ ]:
# ============================================================
# 20. CONFUSION MATRIX
# ============================================================

def save_confusion_matrices(y_true, y_pred, metric_dir, figure_dir):
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    for normalize, suffix in ((None, ""), ("true", "_normalized")):
        values = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES), normalize=normalize)
        pd.DataFrame(values, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(metric_dir / f"confusion_matrix{suffix}.csv")
        fig, ax = plt.subplots(figsize=(8, 7))
        ConfusionMatrixDisplay(values, display_labels=CLASS_NAMES).plot(
            ax=ax, cmap="Blues", values_format="d" if normalize is None else ".2f", colorbar=False)
        plt.xticks(rotation=30, ha="right")
        fig.tight_layout()
        fig.savefig(figure_dir / f"confusion_matrix{suffix}.png", dpi=200, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    return cm


In [ ]:
# ============================================================
# 21. CLASSIFICATION REPORT
# ============================================================

def save_classification_report(y_true, y_pred, metric_dir):
    report = pd.DataFrame(classification_report(y_true, y_pred, labels=range(NUM_CLASSES),
        target_names=CLASS_NAMES, output_dict=True, zero_division=0)).transpose()
    report.to_csv(metric_dir / "classification_report.csv")
    result = {"accuracy": float(accuracy_score(y_true, y_pred)),
              "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred))}
    for average in ("macro", "weighted"):
        for name, function in (("precision", precision_score), ("recall", recall_score), ("f1", f1_score)):
            result[f"{average}_{name}"] = float(function(y_true, y_pred, labels=range(NUM_CLASSES),
                average=average, zero_division=0))
    return report, result


In [ ]:
# ============================================================
# 22. PER-CLASS METRICS
# ============================================================

def save_per_class_table(report, metric_dir):
    table = report.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]].copy()
    table["support"] = table["support"].astype(int)
    table.to_csv(metric_dir / "per_class_metrics.csv", index_label="class")
    display(table)
    return table


In [ ]:
# ============================================================
# 23. SAVE FINAL METRICS — SHARED SCHEMA
# ============================================================

def make_metrics_document(seed, test_result=None, efficiency=None):
    seed_dir = OUT_DIR / f"seed_{seed}"
    training = json.loads((seed_dir / "training_summary.json").read_text(encoding="utf-8"))
    return {
        "experiment": {"experiment_id": EXPERIMENT_ID, "date": experiment_config["created_at"],
                       "member": experiment_config["member"], "dataset_version": DATASET_VERSION},
        "model": {"model_name": MODEL_NAME, "backbone": MODEL_NAME, "weights": PRETRAINED_WEIGHTS,
                  "training_mode": TRAINING_MODE, "fine_tune": FINE_TUNE,
                  **{key: training[key] for key in ("total_parameters", "trainable_parameters", "non_trainable_parameters")}},
        "training_config": {"seed": seed, "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
                            "max_epochs": MAX_EPOCHS, "optimizer": OPTIMIZER,
                            "initial_learning_rate": INITIAL_LR, "loss": LOSS},
        "training_result": {"actual_epochs": training["actual_epochs"], "best_epoch": training["best_epoch"],
                            "best_val_accuracy": training["maximum_val_accuracy_history"],
                            "best_val_loss": training["best_val_loss_history"],
                            "training_time_seconds": training["training_time_seconds"]},
        "test_result": test_result,
        "efficiency": efficiency or {"model_size_mb": training["best_weights_size_mb"],
                                    "inference_time_ms_per_image": None, "fps": None},
        "classes": CLASS_NAMES,
    }


def save_metrics_document(seed, document):
    metric_dir = OUT_DIR / f"seed_{seed}" / "metrics"
    metric_dir.mkdir(parents=True, exist_ok=True)
    (metric_dir / "metrics.json").write_text(json.dumps(document, indent=2), encoding="utf-8")


In [ ]:
# ============================================================
# 24. INFERENCE SPEED
# ============================================================

def inference_efficiency(checkpoint, elapsed, count):
    return {"model_size_mb": checkpoint.stat().st_size / (1024 ** 2),
            "inference_time_ms_per_image": elapsed / count * 1000,
            "fps": count / elapsed}

# Model size means the serialized best weights (not a full optimizer/model bundle).
# Timing uses the configured batch size and distribution strategy, including tf.data I/O.


In [ ]:
# ============================================================
# 25. ERROR ANALYSIS
# ============================================================

def save_error_analysis(predictions, cm, per_class, error_dir):
    wrong = predictions.loc[~predictions["correct"]].sort_values("confidence", ascending=False)
    low_correct = predictions.loc[predictions["correct"]].sort_values("confidence")
    wrong.head(ERROR_SAMPLE_COUNT).to_csv(error_dir / "highest_confidence_wrong.csv", index=False)
    low_correct.head(ERROR_SAMPLE_COUNT).to_csv(error_dir / "lowest_confidence_correct.csv", index=False)
    pairs = [{"true_class": CLASS_NAMES[i], "predicted_class": CLASS_NAMES[j], "count": int(cm[i, j])}
             for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
    pairs.sort(key=lambda row: row["count"], reverse=True)
    pd.DataFrame(pairs, columns=["true_class", "predicted_class", "count"]).to_csv(error_dir / "confusion_pairs.csv", index=False)
    disease_ids = [CLASS_NAMES.index(name) for name in ("anthracnose", "leaf_miner", "red_rust")]
    notes = ["# Failure analysis", f"Misclassified images: {len(wrong)} / {len(predictions)}",
             f"Most frequent confusion: {pairs[0] if pairs else 'None'}",
             f"Lowest recall: {per_class['recall'].idxmin()}",
             f"Lowest precision: {per_class['precision'].idxmin()}"]
    errors_by_class = cm.sum(axis=1) - np.diag(cm)
    notes.append(f"Most errors by true class: {CLASS_NAMES[int(np.argmax(errors_by_class))] if len(wrong) else 'None'}")
    for name in ("healthy", "not_cashew_leaf"):
        count = int(cm[CLASS_NAMES.index(name), disease_ids].sum())
        notes.append(f"{name} predicted as disease: {count}")
    notes += ["", "Visual review pending: inspect saved images for small lesions, blur, lighting, background or similar symptoms.",
              "These are review prompts, not inferred causes. Test findings must not be used to retune this experiment."]
    (error_dir / "failure_analysis.md").write_text("\n".join(notes), encoding="utf-8")
    print("\n".join(notes))
    return wrong, low_correct


In [ ]:
# ============================================================
# 26. SAMPLE PREDICTIONS
# ============================================================

def save_prediction_grid(rows, destination, title):
    if rows.empty:
        destination.with_suffix(".txt").write_text("No examples for this category.", encoding="utf-8")
        return
    count = len(rows)
    columns = min(5, count)
    fig, axes = plt.subplots(int(np.ceil(count / columns)), columns,
                             figsize=(4 * columns, 4 * int(np.ceil(count / columns))), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, row) in zip(axes.flat, rows.iterrows()):
        with Image.open(row["filename"]) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(f"True: {row['true_label']}\nPred: {row['predicted_label']}\nConfidence: {row['confidence']:.3f}", fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(destination, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
# ============================================================
# 17–26. FINAL TEST DRIVER — LOCKED BY DEFAULT
# ============================================================

test_result_rows = []
if not RUN_FINAL_TEST:
    print("FINAL TEST IS LOCKED. No test evaluation or predictions run.")
    for seed in SEEDS:
        metrics_path = OUT_DIR / f"seed_{seed}" / "metrics" / "metrics.json"
        if not metrics_path.exists():
            save_metrics_document(seed, make_metrics_document(seed))
else:
    # Preflight all seeds before reading test data for model evaluation.
    for seed in SEEDS:
        seed_dir = OUT_DIR / f"seed_{seed}"
        for required in (seed_dir / "training_summary.json", seed_dir / "experiment_config.json",
                         seed_dir / "checkpoints" / f"RESNET50_SCRATCH_seed_{seed}_best.weights.h5"):
            if not required.exists():
                raise FileNotFoundError(required)
        saved_seed_config = json.loads((seed_dir / "experiment_config.json").read_text(encoding="utf-8"))
        expected_seed_config = copy.deepcopy(experiment_config)
        expected_seed_config["training"]["seed"] = seed
        if saved_seed_config != expected_seed_config:
            raise ValueError(f"Seed {seed} checkpoint configuration differs from the frozen experiment.")
        existing_metrics = seed_dir / "metrics" / "metrics.json"
        if existing_metrics.exists() and json.loads(existing_metrics.read_text(encoding="utf-8"))["test_result"] is not None:
            raise FileExistsError(f"Final test already recorded for seed {seed}; do not overwrite it.")
    test_ds = make_dataset(test_paths, test_labels, seed=SEEDS[0], training=False)
    for seed in SEEDS:
        seed_dir = OUT_DIR / f"seed_{seed}"
        metric_dir, figure_dir = seed_dir / "metrics", seed_dir / "figures"
        prediction_dir = seed_dir / "predictions"
        error_dir = prediction_dir / "error_cases"
        for folder in (metric_dir, figure_dir, prediction_dir, error_dir):
            folder.mkdir(parents=True, exist_ok=True)
        model, checkpoint = load_best_model(seed)
        test_loss, keras_accuracy = evaluate_test(model, test_ds)
        probabilities, elapsed = predict_test(model, test_ds)
        y_true = np.asarray(test_labels, dtype=np.int32)
        y_pred = probabilities.argmax(axis=1)
        predictions = pd.DataFrame({"filename": test_meta["path"].to_numpy(),
            "true_label": [CLASS_NAMES[i] for i in y_true],
            "predicted_label": [CLASS_NAMES[i] for i in y_pred],
            "confidence": probabilities.max(axis=1), "correct": y_true == y_pred})
        for index, name in enumerate(CLASS_NAMES):
            predictions[f"probability_{name}"] = probabilities[:, index]
        predictions.to_csv(prediction_dir / "predictions.csv", index=False)
        cm = save_confusion_matrices(y_true, y_pred, metric_dir, figure_dir)
        report, result = save_classification_report(y_true, y_pred, metric_dir)
        result["test_loss"] = test_loss
        if abs(keras_accuracy - result["accuracy"]) > 1e-5:
            raise ValueError("Keras accuracy disagrees with ordered predictions.")
        per_class = save_per_class_table(report, metric_dir)
        efficiency = inference_efficiency(checkpoint, elapsed, len(y_true))
        wrong, low_correct = save_error_analysis(predictions, cm, per_class, error_dir)
        save_prediction_grid(wrong.head(ERROR_SAMPLE_COUNT), error_dir / "incorrect_predictions.png", f"Seed {seed}: errors")
        samples = pd.concat([low_correct.head(SAMPLE_COUNT_PER_OUTCOME), wrong.head(SAMPLE_COUNT_PER_OUTCOME)])
        save_prediction_grid(samples, figure_dir / "sample_predictions.png", f"Seed {seed}: correct and incorrect")
        save_metrics_document(seed, make_metrics_document(seed, result, efficiency))
        test_result_rows.append({"seed": seed, **result, **efficiency})
        del model, probabilities
        tf.keras.backend.clear_session()
        gc.collect()
    test_results_df = pd.DataFrame(test_result_rows)
    test_results_df.to_csv(AGGREGATE_DIR / "aggregate_test_results.csv", index=False)
    display(test_results_df)


In [ ]:
# ============================================================
# 26a. TEST MEAN ± STD AND THESIS TABLE
# ============================================================

if not RUN_FINAL_TEST:

    print(
        "Skipped: Final Test chưa chạy."
    )

else:

    summary_metrics = [
        "test_loss",
        "weighted_precision",
        "weighted_recall",
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "balanced_accuracy",
        "weighted_f1",
        "inference_time_ms_per_image",
        "fps",
    ]

    summary_rows = []

    for metric in summary_metrics:

        values = test_results_df[metric]

        summary_rows.append({
            "metric": metric,
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)),
            "report": (
                f"{values.mean():.6f} ± "
                f"{values.std(ddof=1):.6f}"
            ),
        })

    mean_std_df = pd.DataFrame(
        summary_rows
    )

    display(mean_std_df)

    mean_std_df.to_csv(
        AGGREGATE_DIR /
        "test_mean_std_summary.csv",
        index=False
    )

    thesis_metrics = [
        "weighted_precision",
        "weighted_recall",
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "balanced_accuracy",
        "weighted_f1",
    ]

    thesis_rows = []

    for metric in thesis_metrics:

        values = (
            test_results_df[metric]
            * 100
        )

        thesis_rows.append({
            "Metric": metric,
            "Mean (%)": float(values.mean()),
            "Std (%)": float(values.std(ddof=1)),
            "Report": (
                f"{values.mean():.2f} ± "
                f"{values.std(ddof=1):.2f}%"
            ),
        })

    thesis_summary_df = pd.DataFrame(
        thesis_rows
    )

    print("\nThesis-ready table:")
    display(thesis_summary_df)

    thesis_summary_df.to_csv(
        AGGREGATE_DIR /
        "thesis_test_mean_std_summary.csv",
        index=False
    )

    aggregate_metrics = {
        "experiment_id": EXPERIMENT_ID,
        "model": MODEL_NAME,
        "training_mode": TRAINING_MODE,
        "dataset": DATASET_VERSION,
        "seeds": SEEDS,
        "num_seeds": len(SEEDS),
        "std_definition": "sample standard deviation (ddof=1)",
        "test_set_locked": True,
        "per_seed_results": test_result_rows,
        "summary": {
            row["metric"]: {
                "mean": float(row["mean"]),
                "std": float(row["std"]),
            }
            for row in summary_rows
        },
    }

    (
        AGGREGATE_DIR /
        "aggregate_test_metrics.json"
    ).write_text(
        json.dumps(
            aggregate_metrics,
            indent=2
        ),
        encoding="utf-8"
    )


In [ ]:
# ============================================================
# 26b. PER-CLASS MEAN ± STD
# ============================================================

if not RUN_FINAL_TEST:

    print(
        "Skipped: Final Test chưa chạy."
    )

else:

    per_class_rows = []

    for class_name in CLASS_NAMES:

        precision_values = []
        recall_values = []
        f1_values = []

        for seed in SEEDS:

            report_path = (
                OUT_DIR /
                f"seed_{seed}" /
                "metrics" /
                "classification_report.csv"
            )

            report_df = pd.read_csv(
                report_path,
                index_col=0
            )

            precision_values.append(
                float(
                    report_df.loc[
                        class_name,
                        "precision"
                    ]
                )
            )

            recall_values.append(
                float(
                    report_df.loc[
                        class_name,
                        "recall"
                    ]
                )
            )

            f1_values.append(
                float(
                    report_df.loc[
                        class_name,
                        "f1-score"
                    ]
                )
            )

        per_class_rows.append({
            "class": class_name,
            "precision_mean": np.mean(precision_values),
            "precision_std": np.std(precision_values, ddof=1),
            "recall_mean": np.mean(recall_values),
            "recall_std": np.std(recall_values, ddof=1),
            "f1_mean": np.mean(f1_values),
            "f1_std": np.std(f1_values, ddof=1),
            "f1_report": (
                f"{np.mean(f1_values)*100:.2f} ± "
                f"{np.std(f1_values, ddof=1)*100:.2f}%"
            ),
        })

    per_class_summary_df = pd.DataFrame(
        per_class_rows
    )

    display(per_class_summary_df)

    per_class_summary_df.to_csv(
        AGGREGATE_DIR /
        "per_class_test_mean_std.csv",
        index=False
    )


## 27. Optional manual inference
Set RUN_MANUAL_DEMO=True to load seed 42 and enable upload. TensorFlow uses the same RGB conversion and bilinear resize as training. Rescaling is inside the model. The demo seed is fixed in advance.


In [ ]:
# ============================================================
# 27. MANUAL IMAGE INFERENCE
# ============================================================

if RUN_MANUAL_DEMO:
    import ipywidgets as widgets
    demo_weights_path = (
        OUT_DIR /
        f"seed_{DEMO_SEED}" /
        "checkpoints" /
        f"RESNET50_SCRATCH_seed_{DEMO_SEED}_best.weights.h5"
    )
    
    if not demo_weights_path.exists():
        raise FileNotFoundError(
            demo_weights_path
        )
    
    tf.keras.backend.clear_session()
    gc.collect()
    
    with strategy.scope():
        demo_model = build_resnet50_model(
            DEMO_SEED
        )
    
    demo_model.load_weights(
        demo_weights_path
    )
    
    print(
        f"Demo model loaded from predefined seed {DEMO_SEED}"
    )
    print(demo_weights_path)
    
    def predict_uploaded_image(
        model,
        image_bytes,
        filename
    ):
    
        img = Image.open(
            io.BytesIO(image_bytes)
        ).convert("RGB")
    
        img_array = tf.image.resize(
            np.asarray(img, dtype=np.float32), [IMG_SIZE, IMG_SIZE], method="bilinear"
        ).numpy()
    
        batch = np.expand_dims(
            img_array,
            axis=0
        )
    
        # Không /255 ở ngoài vì model có Rescaling(1/255).
        probs = model.predict(
            batch,
            verbose=0
        )[0]
    
        pred_idx = int(
            np.argmax(probs)
        )
    
        confidence = float(
            probs[pred_idx]
        )
    
        result_df = pd.DataFrame({
            "Class": CLASS_NAMES,
            "Probability": probs,
            "Probability (%)": probs * 100,
        }).sort_values(
            "Probability",
            ascending=False
        ).reset_index(drop=True)
    
        print("=" * 60)
        print(f"Image      : {filename}")
        print(f"Prediction : {CLASS_NAMES[pred_idx]}")
        print(f"Confidence : {confidence * 100:.2f}%")
        print(f"Seed       : {DEMO_SEED}")
        print("=" * 60)
    
        display(result_df)
    
        plt.figure(figsize=(7, 7))
        plt.imshow(img)
    
        plt.title(
            f"Prediction: {CLASS_NAMES[pred_idx]}\n"
            f"Confidence: {confidence * 100:.2f}%"
        )
    
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    
    
    uploader = widgets.FileUpload(
        accept=".jpg,.jpeg,.png,.webp",
        multiple=False,
        description="Upload Image"
    )
    
    predict_button = widgets.Button(
        description="Predict",
        button_style="success",
        icon="check"
    )
    
    output = widgets.Output()
    
    display(uploader)
    display(predict_button)
    display(output)
    
    
    def get_uploaded_file(upload_widget):
    
        value = upload_widget.value
    
        if not value:
            return None, None
    
        # ipywidgets 8
        if isinstance(value, (tuple, list)):
    
            uploaded_file = value[0]
    
            filename = uploaded_file["name"]
            content = uploaded_file["content"]
    
            if hasattr(content, "tobytes"):
                content = content.tobytes()
    
            return filename, content
    
        # ipywidgets 7
        if isinstance(value, dict):
    
            filename = list(value.keys())[0]
    
            uploaded_file = value[filename]
            content = uploaded_file["content"]
    
            if hasattr(content, "tobytes"):
                content = content.tobytes()
    
            return filename, content
    
        return None, None
    
    
    def on_predict_clicked(_):
    
        with output:
    
            output.clear_output()
    
            filename, image_bytes = get_uploaded_file(
                uploader
            )
    
            if image_bytes is None:
                print("Vui lòng upload ảnh trước.")
                return
    
            try:
                predict_uploaded_image(
                    demo_model,
                    image_bytes,
                    filename
                )
            except Exception as e:
                print("Error:", e)
    
    
    predict_button.on_click(
        on_predict_clicked
    )
    
    print(
        "Upload một ảnh rồi bấm Predict."
    )
else:
    print("Manual upload demo disabled; set RUN_MANUAL_DEMO=True to enable.")


In [ ]:
# ============================================================
# 28. SAVE README / COMPLETION STATUS / EXPERIMENT REGISTRY
# ============================================================

registry_rows = []
all_complete = True
for seed in SEEDS:
    seed_dir = OUT_DIR / f"seed_{seed}"
    document = json.loads((seed_dir / "metrics" / "metrics.json").read_text(encoding="utf-8"))
    training = document["training_result"]
    test = document["test_result"]
    required = ["model_summary.txt", "experiment_config.json", "logs/history.csv", "logs/learning_curve_analysis.json",
                "metrics/metrics.json", "metrics/classification_report.csv", "metrics/per_class_metrics.csv",
                "metrics/confusion_matrix.csv", "figures/accuracy_curve.png", "figures/loss_curve.png",
                "figures/confusion_matrix.png", "figures/confusion_matrix_normalized.png",
                "figures/sample_predictions.png", "predictions/predictions.csv", "predictions/error_cases/failure_analysis.md",
                f"checkpoints/RESNET50_SCRATCH_seed_{seed}_best.weights.h5"]
    missing = [name for name in required if not (seed_dir / name).exists()]
    complete = not missing and test is not None
    all_complete = all_complete and complete
    status = "READY_FOR_EXPORT" if complete else "INCOMPLETE"
    notes = [f"# {EXPERIMENT_ID} — seed {seed}", f"Status: {status}",
             "", "## Configuration", json.dumps(experiment_config, indent=2, ensure_ascii=False),
             "", "## Training", json.dumps(training, indent=2),
             "", "## Validation", validation_results_df.loc[validation_results_df["seed"] == seed].to_csv(index=False),
             "", "## Test", json.dumps(test, indent=2) if test is not None else "Not evaluated: locked test.",
             "", "## Efficiency", json.dumps(document["efficiency"], indent=2),
             "Weights-only size; batched full-pipeline inference timing, including input processing.",
             "", "## Analysis", "See logs/learning_curve_analysis.json and predictions/error_cases/failure_analysis.md.",
             "Visual failure causes, conclusion and next-experiment decision require researcher review.",
             "Do not use test findings to select this experiment's settings.",
             "", "## Missing artifacts", json.dumps(missing)]
    (seed_dir / "README.md").write_text("\n".join(notes), encoding="utf-8")
    registry_rows.append({"Experiment ID": EXPERIMENT_ID, "Seed": seed, "Model": MODEL_NAME,
        "Dataset": DATASET_VERSION, "IMG Size": IMG_SIZE, "Batch": BATCH_SIZE, "LR": INITIAL_LR,
        "Mode": TRAINING_MODE, "Best Val Acc": training["best_val_accuracy"],
        "Test Acc": test["accuracy"] if test else None, "Macro F1": test["macro_f1"] if test else None,
        "Status": status})

status = "READY_FOR_EXPORT" if all_complete else "INCOMPLETE"
readme = [f"# {EXPERIMENT_ID}", f"Status: {status}",
          f"Dataset: {DATASET_VERSION}; counts: {EXPECTED_TOTALS}", f"Seeds: {SEEDS}",
          "Fixed test membership and class order preserved. No test-based model/seed selection.",
          "", "## Training", training_summary_df.to_csv(index=False),
          "", "## Validation", validation_mean_std_df.to_csv(index=False),
          "", "## Final test", "See per-seed metrics/metrics.json; null means not evaluated.",
          "", "## Reconstructing a checkpoint", "Build build_resnet50_model(seed), then load the corresponding .weights.h5 file.",
          "", "## Result interpretation", "Inspect per-seed curve diagnostics, confusion pairs and error grids.",
          "Record visual findings and scientific conclusions after reviewing outputs; no results are fabricated."]
if (AGGREGATE_DIR / "thesis_test_mean_std_summary.csv").exists():
    readme.append((AGGREGATE_DIR / "thesis_test_mean_std_summary.csv").read_text(encoding="utf-8"))
(OUT_DIR / "README.md").write_text("\n".join(readme), encoding="utf-8")

registry = pd.read_csv(REGISTRY_PATH) if REGISTRY_PATH.exists() else pd.DataFrame()
if not registry.empty:
    if "Experiment ID" not in registry:
        raise ValueError("Existing registry lacks Experiment ID; reconcile its schema before updating.")
    registry = registry.loc[registry["Experiment ID"] != EXPERIMENT_ID]
registry = pd.concat([registry, pd.DataFrame(registry_rows)], ignore_index=True)
registry.to_csv(REGISTRY_PATH, index=False)
registry.to_csv(OUT_DIR / "experiment_registry_snapshot.csv", index=False)
print("Artifacts checked; status:", status)


In [ ]:
# ============================================================
# 29. ZIP RESULTS — FULL AND LIGHTWEIGHT
# ============================================================

FULL_ZIP = OUT_DIR.parent / f"{EXPERIMENT_ID}.zip"
ANALYSIS_ZIP = OUT_DIR.parent / f"{EXPERIMENT_ID}_ANALYSIS.zip"

def create_archive(destination, include_checkpoints):
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUT_DIR.rglob("*")):
            if path.is_file():
                relative = path.relative_to(OUT_DIR)
                if include_checkpoints or "checkpoints" not in relative.parts:
                    archive.write(path, arcname=relative.as_posix())
    with zipfile.ZipFile(destination) as archive:
        if archive.testzip() is not None:
            raise IOError(f"Archive integrity check failed: {destination}")
        archived = set(archive.namelist())
        for seed in SEEDS:
            required = [f"seed_{seed}/logs/history.csv", f"seed_{seed}/metrics/metrics.json",
                        f"seed_{seed}/experiment_config.json", f"seed_{seed}/README.md"]
            if include_checkpoints:
                required.append(f"seed_{seed}/checkpoints/RESNET50_SCRATCH_seed_{seed}_best.weights.h5")
            if not set(required).issubset(archived):
                raise ValueError(f"Archive is missing required seed {seed} artifacts.")
    print(destination, f"({destination.stat().st_size / 1024**2:.2f} MB)")

create_archive(FULL_ZIP, True)
create_archive(ANALYSIS_ZIP, False)
# Only mark complete after outputs exist and both archives passed integrity checks.
if all_complete:
    status = "COMPLETED"
    for path in [OUT_DIR / "README.md"] + [OUT_DIR / f"seed_{seed}" / "README.md" for seed in SEEDS]:
        path.write_text(path.read_text(encoding="utf-8").replace("Status: READY_FOR_EXPORT", "Status: COMPLETED"), encoding="utf-8")
    registry.loc[registry["Experiment ID"] == EXPERIMENT_ID, "Status"] = status
    registry.to_csv(REGISTRY_PATH, index=False)
    registry.to_csv(OUT_DIR / "experiment_registry_snapshot.csv", index=False)
    create_archive(FULL_ZIP, True)
    create_archive(ANALYSIS_ZIP, False)
print("Experiment status:", status)


# EXPERIMENT RESULT SUMMARY

Results are generated at execution; no prior cell outputs are retained.

- Training and efficiency: aggregate/training_summary_5seeds.csv and seed_*/metrics/metrics.json.
- Validation: aggregate/validation_mean_std_summary.csv.
- Final test: aggregate/thesis_test_mean_std_summary.csv, only after explicit unlock.
- Per-class performance: aggregate/per_class_test_mean_std.csv.
- Main confusions: seed_*/predictions/error_cases/confusion_pairs.csv and failure_analysis.md.
- Learning curves: seed_*/figures/ and logs/learning_curve_analysis.json.

## Researcher interpretation (complete after execution)
Record whether train/validation gaps and loss curves suggest overfitting, inspect the error-image grids,
and document visible failure causes with filenames. Record the conclusion and next-experiment decision
using validation evidence. Do not treat test results as tuning feedback.

## Experiment completion checklist
- [ ] Experiment ID, member, dataset version, seeds, environment and GPU recorded.
- [ ] Fixed counts, classes and split fingerprints verified.
- [ ] Five model summaries, best checkpoints, configs and history CSVs saved.
- [ ] Actual/best epochs, LR changes, training time and parameter counts recorded.
- [ ] Accuracy/loss curves reviewed for generalization gaps and early stopping.
- [ ] Configuration frozen before unlocking final test.
- [ ] Test loss, accuracy, macro/weighted metrics and balanced accuracy saved.
- [ ] Raw/normalized confusion matrices and per-class reports saved.
- [ ] Probabilities, error cases and sample images saved and reviewed.
- [ ] Model size, warmed-up inference time and FPS recorded.
- [ ] README, common metrics schema and registry verified.
- [ ] Full ZIP includes best weights; both archives pass integrity checks.
- [ ] Scientific conclusion and next step documented by researcher.

A training-only run remains INCOMPLETE until final evaluation and required outputs are available.
